# Minutely Volume Profiles (21-day Rolling)

This notebook reads minute-bar parquet data from `mana_data/bar_data_parquet`, computes a 21-day backward-looking rolling average and variance of per-minute volumes per symbol and date, and saves the results to `mana_data/minutely_profiles/sym={SYM}/date={YYYYMMDD}/minutely.parquet`.

It also computes a Parkinson-based minute volatility over the same rolling window, plots sample profiles, and provides a small Dash app to browse the outputs by symbol and date.


In [ ]:
import os
from pathlib import Path
from datetime import datetime, date, timedelta
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

# Configuration
DATA_ROOT = "."
INPUT_DATASET = DATA_ROOT / "bar_data_parquet"
OUTPUT_ROOT = DATA_ROOT / "minutely_profiles"
ROLLING_DAYS = 21
MARKET_OPEN_MINUTE = 9 * 60 + 30
TRADING_MINUTES = 390  # 9:30 to 16:00

# Ensure output root exists
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Input dataset: {INPUT_DATASET}")
print(f"Output root  : {OUTPUT_ROOT}")


In [ ]:
from collections import deque

def list_symbols(input_root: Path) -> List[str]:
    syms = []
    for p in sorted((input_root).glob("sym=*")):
        syms.append(p.name.split("=", 1)[1])
    return syms


def list_dates_for_symbol(input_root: Path, symbol: str) -> List[str]:
    sym_dir = input_root / f"sym={symbol}"
    dates = []
    for p in sorted(sym_dir.glob("date=*")):
        dates.append(p.name.split("=", 1)[1])
    return dates


def minute_index_frame() -> pd.DataFrame:
    # Create 390 rows from 9:30 to 16:00 (exclusive of 16:01)
    minutes = []
    for i in range(TRADING_MINUTES):
        tot = MARKET_OPEN_MINUTE + i
        hh = tot // 60
        mm = tot % 60
        minutes.append(f"{hh:02d}:{mm:02d}")
    return pd.DataFrame({"minute": minutes})


def load_day(input_root: Path, symbol: str, date_str: str) -> pd.DataFrame:
    # Reads one day's parquet and returns a DataFrame with columns: minute, trade_volume, trade_high, trade_low, spread
    fp = input_root / f"sym={symbol}" / f"date={date_str}" / "data.parquet"
    table = pq.read_table(fp)
    df = table.to_pandas()

    # Expect standard columns
    expected_cols = {"time", "trade_volume", "trade_high", "trade_low", "bid_price", "ask_price"}
    missing = expected_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns {missing} in {fp}")

    # to datetime and filter regular session
    t = pd.to_datetime(df["time"])
    minutes_from_open = t.dt.hour * 60 + t.dt.minute - MARKET_OPEN_MINUTE
    mask = (minutes_from_open >= 0) & (minutes_from_open < TRADING_MINUTES)
    df = df.loc[mask, ["trade_volume", "trade_high", "trade_low", "bid_price", "ask_price"]].copy()
    df["minute_idx"] = minutes_from_open.loc[mask].astype(int).to_numpy()

    # build output aligned on 390 minutes
    out = minute_index_frame()
    out["minute_idx"] = np.arange(TRADING_MINUTES)
    out = out.merge(df, on="minute_idx", how="left")

    # fill NaNs with 0 for volume; for price fields forward/backfill within day
    out["trade_volume"] = out["trade_volume"].fillna(0)
    for col in ["trade_high", "trade_low", "bid_price", "ask_price"]:
        out[col] = out[col].ffill().bfill()

    # compute per-minute bid/offer spread
    out["spread"] = (out["ask_price"] - out["bid_price"]).astype(float)

    return out[["minute", "minute_idx", "trade_volume", "trade_high", "trade_low", "spread"]]


def parkinson_vol_component(high: pd.Series, low: pd.Series) -> pd.Series:
    # Component per observation: (ln(H/L))^2
    # We'll aggregate mean over window later and apply 1/(4 ln 2) factor with sqrt.
    ratio = (high / low).clip(lower=1e-12)
    return np.log(ratio) ** 2


def compute_and_write_profiles_for_symbol(symbol: str, input_root: Path = INPUT_DATASET, output_root: Path = OUTPUT_ROOT,
                                           rolling_days: int = ROLLING_DAYS) -> List[Tuple[str, Path]]:
    dates = list_dates_for_symbol(input_root, symbol)
    if not dates:
        return []

    # Process by calendar year to reset rolling at year start
    by_year: Dict[str, List[str]] = {}
    for d in dates:
        y = d[:4]
        by_year.setdefault(y, []).append(d)

    written: List[Tuple[str, Path]] = []

    for y, year_dates in by_year.items():
        # Ensure chronological order
        year_dates = sorted(year_dates)

        # Rolling deques per minute for volume, parkinson components, and spread
        # We will maintain arrays of shape (TRADING_MINUTES,) counting accumulated values
        vol_hist: deque[pd.Series] = deque(maxlen=rolling_days)
        hl_comp_hist: deque[pd.Series] = deque(maxlen=rolling_days)
        spread_hist: deque[pd.Series] = deque(maxlen=rolling_days)

        for d in year_dates:
            day_df = load_day(input_root, symbol, d)

            # Append today's series
            vol_hist.append(day_df["trade_volume"].astype(float))
            hl_comp_hist.append(parkinson_vol_component(day_df["trade_high"], day_df["trade_low"]))

            # Build rolling window DataFrame
            vol_window = pd.concat(list(vol_hist), axis=1)
            # Average and variance across days for each minute
            shares_avg = vol_window.mean(axis=1)
            shares_var = vol_window.var(axis=1, ddof=1) if vol_window.shape[1] > 1 else pd.Series(np.zeros(TRADING_MINUTES), index=vol_window.index)

            # Parkinson volatility across window: sqrt( (1/(4 ln 2)) * mean( (ln(H/L))^2 ) )
            hl_window = pd.concat(list(hl_comp_hist), axis=1)
            hl_mean = hl_window.mean(axis=1)
            parkinson_const = 1.0 / (4.0 * np.log(2.0))
            volatility = np.sqrt(parkinson_const * hl_mean).astype(float)

            # Spread rolling stats
            if "spread_hist" not in locals():
                spread_hist = deque(maxlen=rolling_days)
            spread_hist.append(day_df["spread"].astype(float))
            spread_window = pd.concat(list(spread_hist), axis=1)
            spread_avg = spread_window.mean(axis=1)
            spread_var = spread_window.var(axis=1, ddof=1) if spread_window.shape[1] > 1 else pd.Series(np.zeros(TRADING_MINUTES), index=spread_window.index)

            out = pd.DataFrame({
                "minute": day_df["minute"].values,
                "shares_avg": shares_avg.values,
                "shares_variance": shares_var.fillna(0).values,
                "volatility": volatility.values,
                "spread_avg": spread_avg.values,
                "spread_variance": spread_var.fillna(0).values,
            })

            # Write to parquet path sym={sym}/date={YYYYMMDD}/minutely.parquet
            out_dir = output_root / f"sym={symbol}" / f"date={d}"
            out_dir.mkdir(parents=True, exist_ok=True)
            out_fp = out_dir / "minutely.parquet"
            out.to_parquet(out_fp, index=False)
            written.append((d, out_fp))

    return written


In [ ]:
# Run for a subset of symbols for testing
SYMBOLS_SAMPLE = ["AAPL", "MSFT", "GOOGL"]

all_syms = list_symbols(INPUT_DATASET)
print(f"Found {len(all_syms)} symbols. Example: {all_syms[:5]}")

selected_syms = [s for s in SYMBOLS_SAMPLE if (INPUT_DATASET / f"sym={s}").exists()]
print("Selected symbols:", selected_syms)

results_index: Dict[str, List[Tuple[str, Path]]] = {}
for sym in selected_syms:
    print(f"Processing {sym}...")
    res = compute_and_write_profiles_for_symbol(sym)
    print(f"  Wrote {len(res)} daily profile files for {sym}")
    results_index[sym] = res

print("Done subset generation.")


In [ ]:
# Plot sample profiles
import matplotlib.pyplot as plt


def plot_profile(symbol: str, date_str: str):
    fp = OUTPUT_ROOT / f"sym={symbol}" / f"date={date_str}" / "minutely.parquet"
    if not fp.exists():
        print(f"Not found: {fp}")
        return
    df = pd.read_parquet(fp)
    fig, ax1 = plt.subplots(figsize=(12, 4))
    ax1.plot(df["minute"], df["shares_avg"], label="shares_avg")
    ax1.fill_between(np.arange(len(df)), df["shares_avg"], alpha=0.2)
    ax1.set_ylabel("Avg shares")
    ax1.set_xlabel("Minute")
    ax1.tick_params(axis='x', labelrotation=90)

    ax2 = ax1.twinx()
    ax2.plot(df["minute"], df["volatility"], color="tab:red", label="volatility")
    ax2.plot(df["minute"], df.get("spread_avg", pd.Series(np.nan, index=df.index)), color="tab:green", label="spread_avg", alpha=0.8)
    ax2.set_ylabel("Volatility / Spread")

    ax1.set_title(f"{symbol} {date_str} 21d Rolling Minutely Profile")
    fig.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

# Example: try last date for AAPL if exists
if results_index.get("AAPL"):
    sample_date = results_index["AAPL"][-1][0]
    plot_profile("AAPL", sample_date)


In [ ]:
# Small Dash app to explore profiles by symbol/date
import dash
from dash import dcc, html, Input, Output

# Build options
available_symbols = list_symbols(INPUT_DATASET)

def dates_for(sym):
    return list_dates_for_symbol(INPUT_DATASET, sym)

app = dash.Dash(__name__)
app.layout = html.Div([
    html.H3("Minutely Profiles Browser"),
    html.Div([
        html.Label("Symbol"),
        dcc.Dropdown(options=[{"label": s, "value": s} for s in available_symbols], value=available_symbols[0], id="sym"),
    ], style={"width": "300px"}),
    html.Div([
        html.Label("Date"),
        dcc.Dropdown(id="date"),
    ], style={"width": "300px", "marginTop": 10}),
    dcc.Graph(id="profile-graph"),
])

@app.callback(Output("date", "options"), Output("date", "value"), Input("sym", "value"))
def _update_dates(sym):
    dts = dates_for(sym)
    # Prefer latest
    val = dts[-1] if dts else None
    return ([{"label": d, "value": d} for d in dts], val)

@app.callback(Output("profile-graph", "figure"), Input("sym", "value"), Input("date", "value"))
def _plot(sym, d):
    import plotly.graph_objects as go
    if not sym or not d:
        return go.Figure()
    fp = OUTPUT_ROOT / f"sym={sym}" / f"date={d}" / "minutely.parquet"
    if not fp.exists():
        # attempt to compute for this symbol on the fly
        compute_and_write_profiles_for_symbol(sym)
    if not fp.exists():
        return go.Figure()
    df = pd.read_parquet(fp)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["minute"], y=df["shares_avg"], name="shares_avg", fill="tozeroy"))
    fig.add_trace(go.Scatter(x=df["minute"], y=df["volatility"], name="volatility", yaxis="y2"))
    if "spread_avg" in df.columns:
        fig.add_trace(go.Scatter(x=df["minute"], y=df["spread_avg"], name="spread_avg", yaxis="y2"))
    fig.update_layout(
        xaxis=dict(title="Minute", tickangle=90),
        yaxis=dict(title="Avg Shares"),
        yaxis2=dict(title="Volatility/Spread", overlaying="y", side="right"),
        title=f"{sym} {d} Minutely Profile (21d rolling)",
        height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    return fig

# To run in notebook: uncomment to run server externally if needed
# app.run_server(debug=False, host="127.0.0.1", port=8051)

